# Stage 4. Image Background Correction Pipeline

## Notebook Information

**Purpose:**  
Apply a previously computed background function to field-of-view images.

**Workflow summary:**  
Load configuration, metadata, selected image subset, and background data; then save background-corrected field-of-view images.

**Run scope:**  
Training subset by default, unless another subset is selected in the configuration.

**Reproducibility:**  
Use the configuration file as the source of truth for paths and run parameters. Run sections in order for a fresh execution.

### Authors

| Name | Affiliation |
|---|---|
| Alessandro Ulivi | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |
| Edwin Carreno | Scientific Software Center, Heidelberg |
| Christine Schultz | Scientific Software Center, Heidelberg |
| Name Surname | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |

## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import logging

# Third-party imports
from pathlib import Path
from omegaconf import OmegaConf

# Package imports
from acid.image_processing.background.load_background_function import (
    load_background_function,
)
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.metadata.loading import load_metadata
from acid.utils.metadata.filtering import filter_metadata_by_splits


# ----- Temporal (delete once refactored)
import numpy as np
from pprint import pprint

### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

In [ ]:
CONFIG_PATH_FILE = Path("./config_part4.yaml")

config = OmegaConf.load(file_=CONFIG_PATH_FILE)
# print(OmegaConf.to_yaml(config))

## 2. Input Data Preparation

### 2.1. Create output directories

In [ ]:
output_path, secondary_output_path = create_output_directories(
    output_directory=config["output"]["directory"],
    secondary_output_directory=config["output"]["secondary"]["directory"],
    enable_secondary_output=config["output"]["secondary"]["enabled"],
)

### 2.2 Load metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(config["metadata"])

### 2.3 Select training images

In [ ]:
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=config.metadata.dataset_split
)

In [ ]:
metadata_df.head(3)

### 2.4 Load background data

In [ ]:
backgrounds, background_file_names = load_background_function(
    background_config=config.background_correction,
    metadata_df=metadata_df,
)

In [ ]:
logging.info(f"Background(s) shape: {backgrounds.shape}")
logging.info(f"Background(s) filenames: {background_file_names}")

## 3. Background Correction

### 3.1 Helpers

---
**NOTE:**

Important: They will belong to the package functionality.

---


In [ ]:
# FUTURE: move to src/acid/image_processing/background/metadata_columns.py


def build_metadata_dataframe_column_naming(config):
    sep = config.column_name_separator

    return {
        "illum_correct_df_date_clm_name": config.illum_correct_df_date_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_file_name_clm_name": config.illum_correct_df_file_name_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_method_clm_name": config.illum_correct_df_method_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_offset_clm_name": config.illum_correct_df_offset_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_rescale_clm_name": config.illum_correct_df_rescale_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clipping_clm_name": config.illum_correct_df_clipping_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clip_min_value_clm_name": config.illum_correct_df_clip_min_value_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_clip_max_value_clm_name": config.illum_correct_df_clip_max_value_clm_name.replace(
            "_", sep
        ),
        "illum_correct_df_offset_background_clm_name": config.illum_correct_df_offset_background_clm_name.replace(
            "_", sep
        ),
    }

In [ ]:
# background_correction_columns = build_metadata_dataframe_column_naming(
#     config.background_correction
# )
# background_correction_columns

In [ ]:
# FUTURE: src/acid/io/image_loading.py
from pathlib import Path
import tifffile


def load_tiff(file_path, **kwargs):
    """Load a TIFF image from disk."""
    return tifffile.imread(file_path, **kwargs)


# FUTURE: src/acid/image_processing/background/apply_background_correction.py
def load_field_of_view(filename, config, **kwargs):
    """Load a field-of-view TIFF image from disk."""

    file_path = Path(config.fov_directory) / filename
    logging.debug(f"Load field of view from: {file_path} ")

    try:
        return load_tiff(file_path, **kwargs)
    except Exception as error:
        raise OSError(f"Could not load field of view TIFF: {file_path}") from error

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from datetime import datetime
from pathlib import Path

from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict


def build_image_metadata(field_of_view_file, config):
    field_of_view_path = Path(config.fov_directory) / str(field_of_view_file)

    image_metadata = extract_ometif_imagej_metadata(field_of_view_path)

    processing_metadata = {
        config.proc_img_meta_date_name: datetime.now().strftime(
            config.processing_date_format
        ),
        config.proc_img_meta_dtype_name: config.output_dtype,
    }

    background_correction_metadata = {
        config.illum_corr_method_metadata_entry: config.method,
        config.illum_corr_offset_metadata_entry: config.offset,
        config.illum_corr_rescale_metadata_entry: config.rescale_background,
        config.illum_corr_clipping_metadata_entry: config.clip_corrected_image,
        config.illum_corr_clip_min_value_metadata_entry: config.min_clip_value,
        config.illum_corr_clip_max_value_metadata_entry: config.max_clip_value,
        config.illum_corr_offset_background_metadata_entry: config.offset_background,
    }

    image_metadata.update(imagej_compatible_metadata_dict(processing_metadata))
    image_metadata.update(
        imagej_compatible_metadata_dict(background_correction_metadata)
    )

    return image_metadata

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py

from acid.image_processing.background.load_background_function import (
    BackgroundFunctionStrategy,
)


def get_background_for_fov(metadata_row, backgrounds, config):
    strategy = config.background_function_strategy

    if strategy == BackgroundFunctionStrategy.DATASET:
        return backgrounds

    if strategy == BackgroundFunctionStrategy.WELL:
        well = metadata_row[config.well_column_name]
        return backgrounds[well]

    if strategy == BackgroundFunctionStrategy.GRID_POSITION:
        grid_position = metadata_row[config.gridpos_column_name]
        return backgrounds[grid_position]

    raise ValueError(
        f"Invalid background_function_strategy: {strategy}. "
        "Please select either 1, 2 or 3."
    )

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py
from acid.image_processing.correct_background import correct_background


def correct_background_image(image, background, config):
    """Apply background correction to one field-of-view image."""
    correction_kwargs = {
        "method": config.method,
        "channel_axis": config.channel_axis,
        "offset": config.offset,
        "epsilon": config.epsilon,
        "working_dtype": config.working_dtype,
        "output_dtype": config.output_dtype,
        "rescale_background": config.rescale_background,
        "clip_corrected_image": config.clip_corrected_image,
        "min_clip_value": config.min_clip_value,
        "max_clip_value": config.max_clip_value,
        "zero_kwargs": config.zero_kwargs,
        "offset_background": config.offset_background,
        "verbose": config.verbose,
    }

    return correct_background(
        image=image,
        background=background,
        **correction_kwargs,
    )

In [ ]:
# FUTURE: move to src/acid/image_processing/background/apply_background_correction.py
from pathlib import Path


def make_output_filename(field_of_view_file, config):
    """Create the output filename for a background-corrected field of view."""
    field_of_view_file = Path(field_of_view_file)

    suffix = config.ome_suffix
    stem = field_of_view_file.name.removesuffix(suffix)
    print(f"Stem: {stem}")

    return (
        f"{stem}"
        f"{config.save_file_name_separator}"
        f"{config.fov_illumin_corrected_savingword}"
        f"{suffix}"
    )

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py
from pathlib import Path

from acid.utils.save_image import tifffile_save_ometiff


def save_corrected_image(output_filename, corrected_image, image_metadata, config):
    """Save one background-corrected field-of-view image."""

    output_path = Path(config.output_directory) / str(output_filename)

    output_path.parent.mkdir(parents=True, exist_ok=True)

    tifffile_save_ometiff(
        output_path,
        data=corrected_image,
        imagej=config.save_imagej_compatible,
        photometric=config.photometric,
        metadata=image_metadata,
    )

    return output_path

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py
from datetime import datetime


def make_success_result(row_index, field_of_view_file, output_file, config):

    current_date = datetime.now().strftime(config.illum_correct_df_meta_date_format)

    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.illum_correct_df_date_clm_name: current_date,
        config.illum_correct_df_file_name_clm_name: output_file,
        config.illum_correct_df_method_clm_name: config.method,
        config.illum_correct_df_offset_clm_name: config.offset,
        config.illum_correct_df_rescale_clm_name: config.rescale_background,
        config.illum_correct_df_clipping_clm_name: config.clip_corrected_image,
        config.illum_correct_df_clip_min_value_clm_name: config.min_clip_value,
        config.illum_correct_df_clip_max_value_clm_name: config.max_clip_value,
        config.illum_correct_df_offset_background_clm_name: config.offset_background,
    }

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py


def make_failure_result(row_index, field_of_view_file, error, config, stage=None):
    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
        config.illum_correct_df_date_clm_name: config.null_value,
        config.illum_correct_df_file_name_clm_name: config.null_value,
        config.illum_correct_df_method_clm_name: config.null_value,
        config.illum_correct_df_offset_clm_name: config.null_value,
        config.illum_correct_df_rescale_clm_name: config.null_value,
        config.illum_correct_df_clipping_clm_name: config.null_value,
        config.illum_correct_df_clip_min_value_clm_name: config.null_value,
        config.illum_correct_df_clip_max_value_clm_name: config.null_value,
        config.illum_correct_df_offset_background_clm_name: config.null_value,
    }

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py
import pandas as pd


def get_field_of_view_file(metadata_row, config):
    """Return the FOV filename from a metadata row.

    Raises:
        ValueError: If the FOV filename cell is empty.
    """
    field_of_view_file = metadata_row.get(config.fov_column_name)

    if pd.isna(field_of_view_file) or str(field_of_view_file).strip() == "":
        raise ValueError(f"Missing FOV filename in column {config.fov_column_name!r}")

    return str(field_of_view_file).strip()

In [ ]:
# FUTURE: src/acid/image_processing/background/apply_background_correction.py


def apply_background_correction_for_fov(row_index, metadata_row, backgrounds, config):
    # 1. Extract the filename from the row
    try:
        field_of_view_file = get_field_of_view_file(metadata_row, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=None,
            error=error,
            config=config,
            stage="get_field_of_view_file",
        )

    # 2. Load the field of view
    try:
        field_of_view = load_field_of_view(field_of_view_file, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="load_field_of_view",
        )

    # 3. Update the image metadata
    image_metadata = build_image_metadata(field_of_view_file, config)

    # 4. Choose the background function depending on the strategy
    try:
        background = get_background_for_fov(metadata_row, backgrounds, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="get_background_for_fov",
        )

    # 5. Applied background correction
    try:
        corrected_image = correct_background_image(
            image=field_of_view,
            background=background,
            config=config,
        )
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="correct_background_image",
        )

    # 6. Save corrected image
    output_filename = make_output_filename(field_of_view_file, config)

    try:
        save_corrected_image(output_filename, corrected_image, image_metadata, config)
    except Exception as error:
        return make_failure_result(
            row_index=row_index,
            field_of_view_file=field_of_view_file,
            error=error,
            config=config,
            stage="save_corrected_image",
        )

    return make_success_result(row_index, field_of_view_file, output_filename, config)

In [ ]:
from tqdm.notebook import tqdm


def apply_background_correction_batch(metadata_df, backgrounds, config, max_rows=None):
    row_indices = metadata_df.index

    if max_rows is not None:
        row_indices = metadata_df.index[:max_rows]

    results = []

    for row_index in tqdm(row_indices, desc="Applying background correction"):
        result = apply_background_correction_for_fov(
            row_index=row_index,
            metadata_row=metadata_df.loc[row_index],
            backgrounds=backgrounds,
            config=config,
        )

        results.append(result)

    return results

In [ ]:
# FUTURE: src/acid/image_processing/background/metadata.py
import pandas as pd

CORRECTION_METADATA_COLUMN_CONFIG_KEYS = (
    "illum_correct_df_date_clm_name",
    "illum_correct_df_file_name_clm_name",
    "illum_correct_df_method_clm_name",
    "illum_correct_df_offset_clm_name",
    "illum_correct_df_rescale_clm_name",
    "illum_correct_df_clipping_clm_name",
    "illum_correct_df_clip_min_value_clm_name",
    "illum_correct_df_clip_max_value_clm_name",
    "illum_correct_df_offset_background_clm_name",
)


def get_correction_metadata_columns(config) -> list[str]:
    return [getattr(config, key) for key in CORRECTION_METADATA_COLUMN_CONFIG_KEYS]

In [ ]:
# FUTURE: src/acid/image_processing/background/metadata.py
import pandas as pd


def update_metadata_with_correction_results(
    metadata_df, results, config, copy_dataframe=True
):
    if copy_dataframe:
        metadata_df = metadata_df.copy()

    if not results:
        return metadata_df

    metadata_columns = get_correction_metadata_columns(config)

    results_df = pd.DataFrame.from_records(results).set_index("row_index")

    missing_result_columns = [
        column for column in metadata_columns if column not in results_df.columns
    ]

    if missing_result_columns:
        raise KeyError(
            f"Correction results are missing metadata columns: {missing_result_columns}"
        )

    updates_df = results_df[metadata_columns]

    missing_columns = [
        column for column in metadata_columns if column not in metadata_df.columns
    ]

    metadata_df = metadata_df.assign(**{column: pd.NA for column in missing_columns})

    metadata_df = metadata_df.astype(dict.fromkeys(metadata_columns, "object"))

    metadata_df.loc[updates_df.index, metadata_columns] = updates_df.to_numpy()

    return metadata_df

In [ ]:
# FUTURE: src/acid/utils/metadata/saving.py
import datetime as dt
from pathlib import Path

import pandas as pd


def build_metadata_dataframe_filename(config, timestamp=None):
    """Build the standard ACID metadata dataframe filename."""
    if timestamp is None:
        timestamp = dt.datetime.now()

    separator = getattr(config, "save_file_name_separator", "_")
    metadata_file_suffix = config.metadata_file_suffix.format(
        save_file_name_separator=separator
    )

    return separator.join(
        [
            timestamp.strftime(config.metadata_date_format),
            config.project_name,
            config.metadata_savingword,
            metadata_file_suffix,
        ]
    )


def build_metadata_dataframe_path(config, timestamp=None):
    """Build the full save path for a metadata dataframe."""
    metadata_directory = getattr(
        config, "metadata_directory", config.metadata.directory
    )
    metadata_filename = build_metadata_dataframe_filename(
        config=config,
        timestamp=timestamp,
    )

    return Path(metadata_directory) / metadata_filename


def save_metadata_dataframe(metadata_df, config, timestamp=None):
    """Save a metadata dataframe as CSV and return the saved path."""
    metadata_path = build_metadata_dataframe_path(
        config=config,
        timestamp=timestamp,
    )

    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    metadata_df.to_csv(
        metadata_path,
        index=getattr(config, "save_csv_index", False),
    )

    return metadata_path

### 3.2 Apply background correction in batch

In [ ]:
results = apply_background_correction_batch(
    metadata_df=metadata_df,
    backgrounds=backgrounds,
    config=config.background_correction,
)


# apply_results_to_metadata_df(metadata_df, results)
# save_metadata_df(metadata_df, config)

In [ ]:
results

### 3.3 Update Processing Metadata dataframe

In [ ]:
metadata_df_updated = update_metadata_with_correction_results(
    metadata_df=metadata_df,
    results=results,
    config=config.background_correction,
    copy_dataframe=True,
)

In [ ]:
metadata_df_updated.info()

In [ ]:
metadata_df.info()

### 3.4 Save metadata file

In [ ]:
metadata_path = save_metadata_dataframe(metadata_df_updated, config)
print(f"Saved metadata dataframe to: {metadata_path}")

# End of the notebook